# Phase 2: Experiment Tracking & Model Registry with MLflow (10 Features)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

In [2]:
df = pd.read_csv("steam.csv")
def has_term(val, target):
    if not isinstance(val, str): return 0
    return 1 if any(t.strip().lower() == target.lower() for t in val.split(';')) else 0
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
df['release_year'] = df['release_year'].fillna(df['release_year'].median())
df['self_published'] = (df['developer'] == df['publisher']).astype(int)
df['is_mac'] = df['platforms'].apply(lambda x: 1 if isinstance(x, str) and 'mac' in x.lower() else 0)
df['is_linux'] = df['platforms'].apply(lambda x: 1 if isinstance(x, str) and 'linux' in x.lower() else 0)
df['is_multiplayer'] = df['categories'].apply(lambda x: has_term(x, 'Multi-player'))
df['is_vr'] = df['categories'].apply(lambda x: has_term(x, 'VR Support'))
df['is_indie'] = df['genres'].apply(lambda x: has_term(x, 'Indie'))
df['is_action'] = df['genres'].apply(lambda x: has_term(x, 'Action'))
df['is_rpg'] = df['genres'].apply(lambda x: has_term(x, 'RPG'))
df['is_adventure'] = df['genres'].apply(lambda x: has_term(x, 'Adventure'))
df['is_early_access'] = df['genres'].apply(lambda x: has_term(x, 'Early Access'))
features = ['average_playtime', 'achievements', 'release_year', 'self_published', 'english', 'is_mac', 'is_multiplayer', 'is_indie', 'is_action', 'is_early_access']
target = 'price'
df_clean = df.dropna(subset=features + [target])
df_clean = df_clean[(df_clean['price'] >= 4.0) & (df_clean['price'] <= 80.0)]
df_clean = df_clean[~((df_clean['price'] <= 6.0) & (df_clean['achievements'] > 50))]

X = df_clean[features]; y = df_clean[target]
df_clean['price_tier'] = pd.cut(df_clean['price'], bins=[0, 15, 40, 100], labels=[1, 2, 3])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df_clean['price_tier'])

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("SteamGamesPricePrediction_v1")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1776625003056, experiment_id='2', last_update_time=1776625003056, lifecycle_stage='active', name='SteamGamesPricePrediction_v1', tags={}, trace_location=None, workspace='default'>

In [4]:
with mlflow.start_run() as run:
    model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    mlflow.sklearn.log_model(model, "model")
    mlflow.register_model(f"runs:/{run.info.run_id}/model", "BestRegressor_v1")

2026/04/20 19:28:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 19:28:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'BestRegressor_v1' already exists. Creating a new version of this model...
2026/04/20 19:28:54 WARNING mlflow.tracking._model_registry.fluent: Run with id ccbd68eea1b74f10afb569cb15b81fd8 has no artifacts at artifact path 'model', registering model based on models:/m-e3b12e0068bd4ff2b88bdb2da9430171 instead
2026/04/20 19:28:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BestRegressor_v1, ver

🏃 View run nimble-sponge-754 at: http://localhost:5000/#/experiments/2/runs/ccbd68eea1b74f10afb569cb15b81fd8
🧪 View experiment at: http://localhost:5000/#/experiments/2


Created version '12' of model 'BestRegressor_v1'.


In [5]:
alpha = 0.05
lr_sm = sm.OLS(y_train, sm.add_constant(X_train)).fit()
conf_interval = lr_sm.conf_int(alpha)
print(conf_interval)

                            0           1
const            -1179.372279 -948.210832
average_playtime     0.000858    0.001071
achievements        -0.000227    0.000637
release_year         0.477558    0.592182
self_published      -1.332385   -0.769882
english             -2.844555   -0.634501
is_mac              -0.674317   -0.097166
is_multiplayer       2.208515    2.899539
is_indie            -3.833388   -3.227754
is_action           -0.141756    0.402095
is_early_access     -0.498370    0.296806


## Interpretation of the Confidence Intervals
The 95% confidence intervals represent the probable bounds for our regression coefficients. By mapping exactly 15 discrete tags (such as explicit VR tracking vs Generic Action tracking, or tracking if the game is self-published alongside specific release years), we can determine exactly which variables mathematically push pricing thresholds past AAA boundaries safely out of random noise overlaps.